# Hyperliquid

Download data from hyperliquid (https://hyperfoundation.org/). Hyperliquid is a DEX.

In [ ]:
#| default_exp hyperliquid

In [ ]:
#|hide
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
#| export
from nbdev.showdoc import *
import json
from typing import List, Dict, Tuple, Optional, Union, Any, Callable
from hyperliquid.utils import constants
import os
import csv

import eth_account
from eth_account.signers.local import LocalAccount
from hyperliquid.exchange import Exchange
from hyperliquid.info import Info
import pandas as pd
from datetime import datetime
import datetime as dt
import numpy as np

## Connect to Hyperliquid API 
### setup function

In [ ]:
#| export
def setup(base_url=None, skip_ws=False, perp_dexs=None,config='../config_hyperliquid.json'):
    # This function is copied from hyperliquid-python-sdk/examples/example_utils.py
    # for setting up the environment in our script.
    # config_path = os.path.join(os.path.dirname(__file__), "config.json")
    config_path = config
    with open(config_path) as f:
        config = json.load(f)
    account: LocalAccount = eth_account.Account.from_key(config["secret_key"])
    address = config["account_address"]
    if address == "":
        address = account.address
    print("Running with account address:", address)
    if address != account.address:
        print("Running with agent address:", account.address)
    info = Info(base_url, skip_ws, perp_dexs=perp_dexs)
    user_state = info.user_state(address)
    spot_user_state = info.spot_user_state(address)
    margin_summary = user_state["marginSummary"]
    if float(margin_summary["accountValue"]) == 0 and len(spot_user_state["balances"]) == 0:
        print("Not running the example because the provided account has no equity.")
        url = info.base_url.split(".", 1)[1]
        error_string = f"No accountValue:\nIf you think this is a mistake, make sure that {address} has a balance on {url}.\nIf address shown is your API wallet address, update the config to specify the address of your account, not the address of the API wallet."
        raise Exception(error_string)
    exchange = Exchange(account, base_url, account_address=address, perp_dexs=perp_dexs)
    return address, info, exchange

### Example

In [ ]:
#| eval:false
address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)

Running with account address: 0x143E18B563C4aD6913a9D89C774fE69A54F66cAa
Running with agent address: 0x0486f56Bf31b2E3B880248dAcd1BFf3C8bdC09e0


## Get historical prerpetual price data
### retrive_hyperliquid_perp_price function

In [ ]:
#| export
def retrieve_hyperliquid_perp_price(coin="ETH", interval="1h", 
                                end_date=datetime.now(dt.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
                                start_date=(datetime.now(dt.timezone.utc)-pd.Timedelta(days=2)).strftime('%Y-%m-%dT%H:%M:%SZ'),
                                info=None):
    """
    Retrieves historical candle data from Hyperliquid for a given coin and time interval.

    Args:
        coin (str, optional): Coin symbol (e.g. "ETH"). Defaults to "ETH".
        interval (str, optional): Candle interval ("1m", "5m", "15m", "1h", "4h", "1d"). Defaults to "1h".
        end_date (str, optional): End datetime in ISO 8601 format. Defaults to current UTC time.
        start_date (str, optional): Start datetime in ISO 8601 format. Defaults to 2 days before end_date.
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.

    Returns:
        pandas.DataFrame: DataFrame containing the OHLCV data with columns:
            - datetime: Timestamp for the candle (UTC)
            - open: Opening price of the interval
            - high: Highest traded price in the interval
            - low: Lowest traded price in the interval
            - close: Closing price of the interval
            - volume: Trading volume in the interval
            - coin: Coin symbol
        Returns None if the API request fails or returns no data.

    Notes:
        - All datetime values are in UTC timezone
        - Requires Hyperliquid Info client to be initialized
    """
    if info is None:
        from hyperliquid.info import Info
        from hyperliquid.utils import constants
        address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)
    try:
        # Convert datetime strings to Unix milliseconds timestamps
        start_dt = pd.to_datetime(start_date)
        end_dt = pd.to_datetime(end_date)
        
        start_time_ms = int(start_dt.timestamp() * 1000)
        end_time_ms = int(end_dt.timestamp() * 1000)
        
        # Get candles from Hyperliquid
        candles = info.candles_snapshot(name=coin, interval=interval, 
                                       startTime=start_time_ms, endTime=end_time_ms)
        
        if not candles:
            return None
        
        # Convert to DataFrame
        df = pd.DataFrame(candles)
        
        # Convert timestamp to datetime
        df['datetime'] = pd.to_datetime(df['t'], unit='ms')
        
        # Rename columns to match coinbase format
        df = df.rename(columns={
            'o': 'open',
            'h': 'high', 
            'l': 'low',
            'c': 'close',
            'v': 'volume'
        })
        
        # Add coin column
        df['coin'] = coin
        
        # Sort by datetime and reorder columns
        df = df.sort_values(by='datetime')
        df = df[['datetime', 'open', 'high', 'low', 'close', 'volume', 'coin']]
        df = df.astype({'open': 'float64', 'high': 'float64', 'low': 'float64', 'close': 'float64', 'volume': 'float64'})
        
        return df.reset_index(drop=True)
        
    except Exception as e:
        print(f"Error retrieving candles for {coin}: {e}")
        return None

### Example

You will need a api key and secret key from the Hyperliquid API. Store this into the `config_hyperliquid.json` file in the same directory as your script. See: https://app.hyperliquid.xyz/API

The best is to first call the `setup` function once to initialize the Hyperliquid Info client. Then pass "info" to any function that requires it. That will avoid calling the `setup` function multiple times.

Typical usage:

In [ ]:
#| eval: false
perp = retrieve_hyperliquid_perp_price(coin="ETH", interval="1h",info=info)
print(perp.head())

             datetime    open    high     low   close       volume coin
0 2025-11-04 15:00:00  3554.9  3586.2  3495.1  3504.8   59114.6342  ETH
1 2025-11-04 16:00:00  3504.8  3519.2  3423.8  3427.7   57387.4211  ETH
2 2025-11-04 17:00:00  3427.7  3427.9  3360.0  3370.7  211754.0933  ETH
3 2025-11-04 18:00:00  3370.7  3371.3  3253.7  3302.2  170831.2548  ETH
4 2025-11-04 19:00:00  3302.1  3315.4  3224.8  3230.4   97127.1005  ETH


## List of spot tickers
### spot_tickers function


In [ ]:
#| export
def spot_tickers(coin="ETH", base='USDC',info=None):
    """
    Retrieves current tickers for a given coin.

    Args:
        coin (str, optional): Coin symbol (e.g. "ETH"). Defaults to "ETH".
        base (str, optional): Coin symbol (e.g. "USDC"). Defaults to "USDC".
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.

    Returns:
        spot ticker (non intuitive symbol)
        Returns None if the API request fails or returns no data.

    Notes:
        - Requires Hyperliquid Info client to be initialized
    """
    if info is None:
        address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)
    maps =info.spot_meta_and_asset_ctxs()
    # change of ticker for ETH and BTC.... they should have U in front of the name.... Hyperliquid's naming convention...
    if coin.upper() == "ETH":
        coin = "UETH"
    elif coin.upper() == "BTC":
        coin = "UBTC"
    elif coin.upper() == "DOGE":
        coin = "UDOGE"
    elif coin.upper() == "SOL":
        coin = "USOL"
    if base.upper() == "USDC":
        base = "USDC"
    # Find the index of the coin and base in the universe of tokens
    # If not found, return None
    id_coin, id_base = None, None
    for token_ctx in maps[0]['tokens']:
        if token_ctx['name'] == coin:
            id_coin = token_ctx['index']
        elif token_ctx['name'] == base:
            id_base = token_ctx['index']
    if id_coin is None or id_base is None:
        return None
    for i in maps[0]['universe']:
        if i['tokens'] == [id_coin,id_base]:
            return i['name']
    id_coin, id_base = None, None
    for token_ctx in maps[0]['tokens']:
        if token_ctx['name'] == coin:
            id_coin = token_ctx['index']
        elif token_ctx['name'] == base:
            id_base = token_ctx['index']
    if id_coin is None or id_base is None:
        return None
    for i in maps['universe']:
        if i['tokens'] == [id_coin,id_base]:
            return i['name']
    return None                

### Example

In [ ]:
#| eval: false
stk = spot_tickers(info=info,coin="ETH",base="USDC")
print(f'spot ticker for ETH: ', stk)

spot ticker for ETH:  @151


## Get historical spot price
### retrieve_hyperliquid_spot_price function

In [ ]:
#| export
def retrieve_hyperliquid_spot_price(coin="ETH", base='USDC',interval="1h", 
                                end_date=datetime.now(dt.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
                                start_date=(datetime.now(dt.timezone.utc)-pd.Timedelta(days=2)).strftime('%Y-%m-%dT%H:%M:%SZ'),
                                info=None,recheck=False):
    # get the ticker for the given coin
    if recheck:
        ticker = spot_tickers(coin=coin,base=base,info=info)
    else:
        spot_dict = {'BTC': '@142',
                        'ETH': '@151',
                        'SOL': '@156',
                        'HYPE': '@107',
                        'TRUMP': '@9',
                        'BERA': '@117',
                        'PUMP': '@20'}
        ticker = spot_dict.get(coin.upper(), None)
        if ticker is None:
            ticker = spot_tickers(coin=coin,base=base,info=info)

    if ticker is None:
        print(f"{coin} is not listed.")
        return pd.DataFrame()
    # get the price for the given ticker... same function used for perpetuals but
    # ticker price is different...
    try:
        price = retrieve_hyperliquid_perp_price(coin=ticker, interval=interval, 
                                end_date=end_date,
                                start_date=start_date,
                                info=info)
        price['coin'] = coin
        return price
    except Exception as e:
        print(f"Error retrieving price for {coin}: {e}")
        return None

### Example

The ticker for Ethereum (ETH) is "@151" ("UETH"). There are other unusual choices but the function handles these choices by changing the ticker inside the function. 

In [ ]:
#| eval: false
spot=retrieve_hyperliquid_spot_price(info=info)
print(spot.tail())

              datetime    open    high     low   close    volume coin
44 2025-11-06 11:00:00  3393.1  3406.3  3381.0  3398.8  127.4400  ETH
45 2025-11-06 12:00:00  3398.7  3402.6  3342.4  3349.4  141.6060  ETH
46 2025-11-06 13:00:00  3350.9  3401.8  3349.8  3390.2  119.1310  ETH
47 2025-11-06 14:00:00  3390.1  3398.5  3320.0  3327.1  396.8991  ETH
48 2025-11-06 15:00:00  3327.6  3345.0  3321.7  3345.0   70.1237  ETH


If you want to find out which ticker is used for a given coin, you can use the `spot_tickers` function. For example, for Ethereum (ETH) quoted in USDC is:

In [ ]:
#| eval: false
stk = spot_tickers(info=info,coin="ETH",base="USDC")
print(f'spot ticker for ETH: ', stk)

spot ticker for ETH:  @151


## List all tokens in Hyperliquid
### hyperliquid_tokens function

In [ ]:
#| export
def hyperliquid_tokens(info=None,rm_delisted=True):
    if info is None:
        from hyperliquid.info import Info
        from hyperliquid.utils import constants
        address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)

    # Get universe details
    tokens = info.meta_and_asset_ctxs()[0].get('universe')
    
    # Create DataFrame with token details
    df = pd.DataFrame(tokens)
    df['isDelisted'] = ~df['isDelisted'].isna()
    df['onlyIsolated'] = ~df['onlyIsolated'].isna()
    df = df.loc[(~df['isDelisted']) & (~df['onlyIsolated'])]
    return df

### Example

In [ ]:
#| eval: false
tokens = hyperliquid_tokens(info)
print(tokens)

     szDecimals  name  maxLeverage  marginTableId  isDelisted  onlyIsolated  \
0             5   BTC           40             56       False         False   
1             4   ETH           25             55       False         False   
2             2  ATOM            5              5       False         False   
4             1  DYDX            5              5       False         False   
5             2   SOL           20             54       False         False   
..          ...   ...          ...            ...         ...           ...   
211           0  HEMI            3              3       False         False   
212           0  APEX            3              3       False         False   
213           0    2Z            3              3       False         False   
214           2   ZEC           10             52       False         False   
216           0   MET            3              3       False         False   

    marginMode  
0          NaN  
1          NaN  


## Get funding rate history
### retrieve_hyperliquid_funding_history function

In [ ]:
#| export
def funding_calc(rate,premium,max_rate=0.0005,min_rate=-0.0005):
    return premium+max(min(rate-premium, max_rate), min_rate)

In [ ]:
#| export
def retrieve_hyperliquid_funding_history(coin="ETH", 
                                        end_date=datetime.now(dt.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
                                        start_date=(datetime.now(dt.timezone.utc)-pd.Timedelta(days=2)).strftime('%Y-%m-%dT%H:%M:%SZ'),
                                        info=None,
                                        calc=False):
    """
    Retrieves funding rate history from Hyperliquid for a given coin and time period.

    Args:
        coin (str, optional): Coin symbol (e.g. "ETH"). Defaults to "ETH".
        end_date (str, optional): End datetime in ISO 8601 format. Defaults to current UTC time.
        start_date (str, optional): Start datetime in ISO 8601 format. Defaults to 7 days before end_date.
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.

    Returns:
        pandas.DataFrame: DataFrame containing the funding history with columns:
            - datetime: Timestamp for the funding rate (UTC)
            - funding_rate: The funding rate value
            - premium: The premium component
            - coin: Coin symbol
        Returns None if the API request fails or returns no data.

    Notes:
        - All datetime values are in UTC timezone
        - Funding rates are typically updated every hour
        - Requires Hyperliquid Info client to be initialized
    """
    if info is None:
        from hyperliquid.info import Info
        from hyperliquid.utils import constants
        address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)
    
    try:
        # Convert datetime strings to Unix milliseconds timestamps
        start_dt = pd.to_datetime(start_date)
        end_dt = pd.to_datetime(end_date)
        
        start_time_ms = int(start_dt.timestamp() * 1000)
        end_time_ms = int(end_dt.timestamp() * 1000)
        
        # Get funding history from Hyperliquid
        funding_rates = info.funding_history(
            name=coin,
            startTime=start_time_ms,
            endTime=end_time_ms
        )
        
        if not funding_rates:
            return None
        
        # Convert to DataFrame
        df = pd.DataFrame(funding_rates)
        
        # Convert timestamp from milliseconds to datetime
        df['datetime'] = pd.to_datetime(df['time'], unit='ms')
        
        # Rename columns for clarity
        df = df.rename(columns={
            'fundingRate': 'funding_rate',
            'premium': 'premium'
        })
        
        # Convert multiple columns to float
        df[['funding_rate', 'premium']] = df[['funding_rate', 'premium']].astype(float)
        # Add coin column
        df['coin'] = coin
        
        # Sort by datetime and reorder columns
        df = df.sort_values(by='datetime')
        df = df[['datetime', 'funding_rate', 'premium', 'coin']]
        
        # Drop the original time column if it exists
        if 'time' in df.columns:
            df = df.drop(columns=['time'])
        
        df['fund_calc'] = df['funding_rate']
        if calc:
            vectorized_funding_calc = np.vectorize(funding_calc)
            df['fund_calc'] = vectorized_funding_calc(df['funding_rate'], df['premium'])
        
        return df.reset_index(drop=True)
        
    except Exception as e:
        print(f"Error retrieving funding history for {coin}: {e}")
        return None

### Example

In [ ]:
#| eval: false
f_r = retrieve_hyperliquid_funding_history(info=info,calc=True)
print(f_r.tail())

                  datetime  funding_rate   premium coin  fund_calc
43 2025-11-06 11:00:00.102      0.000013 -0.000206  ETH   0.000013
44 2025-11-06 12:00:00.043      0.000013 -0.000178  ETH   0.000013
45 2025-11-06 13:00:00.022      0.000013 -0.000255  ETH   0.000012
46 2025-11-06 14:00:00.005      0.000013 -0.000284  ETH   0.000012
47 2025-11-06 15:00:00.021      0.000013 -0.000379  ETH   0.000012


## Unified function for easy data retrieval
### retrieve_hyperliquid_data function

In [ ]:
#| export
def retrieve_hyperliquid_data(ticker="ETH", 
                              data_type="perp",
                              start_date=None,
                              end_date=None,
                              lookback=2,
                              interval="1h",
                              base="USDC",
                              round_to_hour=False,
                              info=None):
    """
    Unified function to retrieve funding rates, perpetual prices, or spot prices from Hyperliquid.
    
    Args:
        ticker (str, optional): Coin symbol (e.g. "ETH", "BTC"). Defaults to "ETH".
        data_type (str, optional): Type of data to retrieve - "funding", "perp", or "spot". Defaults to "perp".
        start_date (str, optional): Start date as string. Can be:
            - ISO format: "2024-01-15T10:30:00Z" or "2024-01-15T10:30:00"
            - Date only: "2024-01-15"
            - If None, calculated from lookback. Defaults to None.
        end_date (str, optional): End date as string (same formats as start_date).
            - If None, uses current UTC time. Defaults to None.
        lookback (int, optional): Number of days to look back from end_date if start_date is None. 
            Defaults to 2.
        interval (str, optional): Candle interval for perp/spot data ("1m", "5m", "15m", "1h", "4h", "1d"). 
            Defaults to "1h". Not used for funding rates.
        base (str, optional): Base currency for spot prices (e.g. "USDC"). Defaults to "USDC".
            Not used for perp or funding rates.
        round_to_hour (bool, optional): If True, rounds start_date and end_date to nearest hour.
            Useful for funding rates which update hourly. Defaults to False.
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.
    
    Returns:
        pandas.DataFrame: DataFrame containing the requested data with appropriate columns:
            - For "funding": datetime, funding_rate, premium, coin
            - For "perp": datetime, open, high, low, close, volume, coin
            - For "spot": datetime, open, high, low, close, volume, coin
        Returns None if the API request fails or returns no data.
    
    Examples:
        # Get 7 days of funding rates for ETH, rounded to hour
        df = retrieve_hyperliquid_data("ETH", "funding", lookback=7, round_to_hour=True, info=info)
        
        # Get perp prices between specific dates with 4h interval
        df = retrieve_hyperliquid_data("BTC", "perp", 
                                      start_date="2024-01-01", 
                                      end_date="2024-01-15",
                                      interval="4h", info=info)
        
        # Get spot prices for last 30 days with 1h interval
        df = retrieve_hyperliquid_data("ETH", "spot", lookback=30, 
                                      interval="1h", base="USDC", info=info)
    
    Notes:
        - All datetime values are in UTC timezone
        - Valid data_type values: "funding", "perp", "spot"
        - Funding rates are updated hourly, so round_to_hour=True is recommended
        - Requires Hyperliquid Info client to be initialized
    """
    # Validate data_type
    valid_types = ["funding", "perp", "spot"]
    if data_type not in valid_types:
        raise ValueError(f"data_type must be one of {valid_types}, got '{data_type}'")
    
    # Initialize info client if not provided
    if info is None:
        from hyperliquid.info import Info
        from hyperliquid.utils import constants
        address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)
    
    # Handle end_date
    if end_date is None:
        end_dt = pd.Timestamp.now(tz='UTC')
    else:
        # Parse end_date string
        try:
            end_dt = pd.to_datetime(end_date)
        except Exception as e:
            print(f"Error parsing end_date '{end_date}': {e}")
            return None

    # Handle start_date
    if start_date is None:
        # Calculate from lookback
        start_dt = end_dt - pd.Timedelta(days=lookback)
    else:
        # Parse start_date string
        try:
            start_dt = pd.to_datetime(start_date)
        except Exception as e:
            print(f"Error parsing start_date '{start_date}': {e}")
            return None
    
    # Convert to ISO format strings
    start_date_str = start_dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    end_date_str = end_dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    
    # Call appropriate function based on data_type
    try:
        if data_type == "funding":
            df = retrieve_hyperliquid_funding_history(
                coin=ticker,
                start_date=start_date_str,
                end_date=end_date_str,
                info=info
                )
            if round_to_hour:
                df['datetime'] = df['datetime'].apply(lambda x: x.round('h'))
            return df
        elif data_type == "perp":
            return retrieve_hyperliquid_perp_price(
                coin=ticker,
                interval=interval,
                start_date=start_date_str,
                end_date=end_date_str,
                info=info
            )
        elif data_type == "spot":
            return retrieve_hyperliquid_spot_price(
                coin=ticker,
                base=base,
                interval=interval,
                start_date=start_date_str,
                end_date=end_date_str,
                info=info
            )
    except Exception as e:
        print(f"Error retrieving {data_type} data for {ticker}: {e}")
        return None

### Example

In [ ]:
#| eval: false
## Example 1: Get funding rates for last 7 days, rounded to hour
funding_df = retrieve_hyperliquid_data(
    ticker="ETH",
    data_type="funding",
    lookback=7,
    round_to_hour=True,
    info=info
)
print(funding_df.tail())

               datetime  funding_rate   premium coin  fund_calc
163 2025-11-06 11:00:00      0.000013 -0.000206  ETH   0.000013
164 2025-11-06 12:00:00      0.000013 -0.000178  ETH   0.000013
165 2025-11-06 13:00:00      0.000013 -0.000255  ETH   0.000013
166 2025-11-06 14:00:00      0.000013 -0.000284  ETH   0.000013
167 2025-11-06 15:00:00      0.000013 -0.000379  ETH   0.000013


In [ ]:
#| eval: false
## Example 2: Get perpetual prices with specific dates and 4h interval
perp_df = retrieve_hyperliquid_data(
    ticker="BTC",
    data_type="perp",
    start_date="2024-01-01",
    end_date="2024-01-15",
    interval="4h",
    info=info
)
print(perp_df.tail())

              datetime     open     high      low    close     volume coin
80 2024-01-14 08:00:00  43034.0  43114.0  42786.0  42862.0  158.38267  BTC
81 2024-01-14 12:00:00  42858.0  43019.0  42743.0  42936.0  261.86230  BTC
82 2024-01-14 16:00:00  42936.0  43035.0  42681.0  42698.0  304.01708  BTC
83 2024-01-14 20:00:00  42699.0  42772.0  41762.0  41786.0  721.09226  BTC
84 2024-01-15 00:00:00  41779.0  42674.0  41737.0  42612.0  281.41095  BTC


In [ ]:
#| eval: false
## Example 3: Get spot prices for last 30 days
spot_df = retrieve_hyperliquid_data(
    ticker="ETH",
    data_type="spot",
    lookback=30,
    interval="1h",
    base="USDC",
    info=info
)
print(spot_df.tail())


               datetime    open    high     low   close    volume coin
716 2025-11-06 11:00:00  3393.1  3406.3  3381.0  3398.8  127.4400  ETH
717 2025-11-06 12:00:00  3398.7  3402.6  3342.4  3349.4  141.6060  ETH
718 2025-11-06 13:00:00  3350.9  3401.8  3349.8  3390.2  119.1310  ETH
719 2025-11-06 14:00:00  3390.1  3398.5  3320.0  3327.1  396.8991  ETH
720 2025-11-06 15:00:00  3327.6  3345.6  3321.7  3345.6   70.2237  ETH


In [ ]:
#| eval: false
## Example 4: Using datetime strings with time information
data_df = retrieve_hyperliquid_data(
    ticker="ETH",
    data_type="perp",
    lookback=2,
    interval="1h",
    info=info
)
print(data_df.tail())

              datetime    open    high     low   close      volume coin
44 2025-11-06 11:00:00  3393.2  3407.3  3379.9  3398.6   7916.9470  ETH
45 2025-11-06 12:00:00  3398.6  3404.6  3341.7  3348.5  19893.1087  ETH
46 2025-11-06 13:00:00  3348.5  3402.3  3348.5  3389.8  19783.4383  ETH
47 2025-11-06 14:00:00  3389.9  3399.8  3318.8  3327.9  32999.1516  ETH
48 2025-11-06 15:00:00  3327.5  3347.3  3320.6  3345.9  30055.8441  ETH


In [ ]:
#| eval: false
## Example 5: Get funding rates with date-only strings
funding_df = retrieve_hyperliquid_data(
    ticker="BTC",
    data_type="funding",
    start_date="2025-08-29",
    end_date="2025-09-01",
    round_to_hour=True,
    info=info
)
print(funding_df)

              datetime  funding_rate   premium coin  fund_calc
0  2025-08-29 00:00:00      0.000013  0.000251  BTC   0.000013
1  2025-08-29 01:00:00      0.000013  0.000234  BTC   0.000013
2  2025-08-29 02:00:00      0.000013  0.000124  BTC   0.000013
3  2025-08-29 03:00:00      0.000013  0.000068  BTC   0.000013
4  2025-08-29 04:00:00      0.000013  0.000086  BTC   0.000013
..                 ...           ...       ...  ...        ...
67 2025-08-31 19:00:00      0.000013 -0.000035  BTC   0.000013
68 2025-08-31 20:00:00      0.000013 -0.000082  BTC   0.000013
69 2025-08-31 21:00:00      0.000013 -0.000028  BTC   0.000013
70 2025-08-31 22:00:00      0.000013 -0.000059  BTC   0.000013
71 2025-08-31 23:00:00      0.000013 -0.000012  BTC   0.000013

[72 rows x 5 columns]


In [ ]:
#| eval: false
data_df = retrieve_hyperliquid_data(
    ticker="ETH",
    data_type="perp",
    start_date="2025-06-15T10:30:00",
    end_date="2025-06-20T15:45:00",
    interval="1h",
    info=info
)
data_df.tail()

,datetime,open,high,low,close,volume,coin
121,2025-06-20 11:00:00,2551.4,2555.0,2540.1,2548.2,27254.2428,ETH
122,2025-06-20 12:00:00,2548.1,2557.0,2546.1,2550.7,15547.1122,ETH
123,2025-06-20 13:00:00,2550.6,2558.4,2527.5,2534.9,32702.0042,ETH
124,2025-06-20 14:00:00,2535.0,2539.6,2488.1,2488.5,49493.5297,ETH
125,2025-06-20 15:00:00,2488.4,2506.8,2486.1,2492.1,42340.1040,ETH


## Order book data
### retrieve_hyperliquid_l2_snapshot

The will return the snapshot of the order book for the specified ticker in the moment you call the function. It does not provide historical data. 

This is good only for perpetual markets.

In [ ]:
def unix_to_datetime(t):
    return datetime.fromtimestamp(t/1000).strftime("%Y-%m-%d %H:%M:%S.%f")
#The funding rates are reset every hour.
#t = datetime.datetime.fromtimestamp(i['time']/1000).strftime("%Y-%m-%d %H:%M:%S.%f")
#print(t)

In [ ]:

#| export
def retrieve_hyperliquid_l2_snapshot(coin="ETH", info=None):
    """
    Retrieves current L2 order book snapshot from Hyperliquid for a given coin.
    
    Args:
        coin (str, optional): Coin symbol (e.g. "ETH", "BTC"). Defaults to "ETH".
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.
    
    Returns:
        pandas.DataFrame: DataFrame containing the order book snapshot with columns:
            - datetime: Timestamp of the snapshot (UTC)
            - side: Order side ("bid" or "ask")
            - price: Price level
            - size: Total size at this price level
            - num_orders: Number of orders at this price level
        Returns None if the API request fails or returns no data.
    
    Notes:
        - This is a snapshot at the moment the function is called
        - All datetime values are in UTC timezone
        - Bids are sorted from highest to lowest price
        - Asks are sorted from lowest to highest price
        - Requires Hyperliquid Info client to be initialized
    """
    if info is None:
        address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)
    
    try:
        # Get L2 snapshot from Hyperliquid
        l2_data = info.l2_snapshot(name=coin)
        
        if not l2_data or 'levels' not in l2_data:
            return None
        
        # Convert timestamp to datetime
        timestamp = pd.to_datetime(l2_data['time'], unit='ms')
        
        # Extract bid and ask levels
        bids = l2_data['levels'][0]  # First element is bids
        asks = l2_data['levels'][1]  # Second element is asks
        
        # Create list to store all rows
        rows = []
        
        # Process bids
        for bid in bids:
            rows.append({
                'datetime': timestamp,
                'side': 'bid',
                'price': float(bid['px']),
                'size': float(bid['sz']),
                'num_orders': int(bid['n'])
            })
        
        # Process asks
        for ask in asks:
            rows.append({
                'datetime': timestamp,
                'side': 'ask',
                'price': float(ask['px']),
                'size': float(ask['sz']),
                'num_orders': int(ask['n'])
            })
        
        # Create DataFrame
        df = pd.DataFrame(rows)
        
        # Reorder columns for consistency
        df = df[['datetime', 'side', 'price', 'size', 'num_orders']]
        
        return df
        
    except Exception as e:
        print(f"Error retrieving L2 snapshot for {coin}: {e}")
        return None

### Example

In [ ]:
#| eval: false

# Example usage: Get L2 order book snapshot for ETH
l2_snapshot = retrieve_hyperliquid_l2_snapshot(coin="ETH", info=info)
print(l2_snapshot.head(5))
print(l2_snapshot.tail(5))

# Check the structure
print(f"\nTotal levels: {len(l2_snapshot)}")
print(f"Bids: {len(l2_snapshot[l2_snapshot['side'] == 'bid'])}")
print(f"Asks: {len(l2_snapshot[l2_snapshot['side'] == 'ask'])}")
print(f"Snapshot time: {l2_snapshot['datetime'].iloc[0]}")

                 datetime side   price      size  num_orders
0 2025-11-06 15:10:33.501  bid  3344.7   17.7173           8
1 2025-11-06 15:10:33.501  bid  3344.6  145.0540           4
2 2025-11-06 15:10:33.501  bid  3344.5  144.9732           9
3 2025-11-06 15:10:33.501  bid  3344.4  194.7901           6
4 2025-11-06 15:10:33.501  bid  3344.3   60.8754           3
                  datetime side   price      size  num_orders
35 2025-11-06 15:10:33.501  ask  3347.0   26.6027           3
36 2025-11-06 15:10:33.501  ask  3347.1  144.3073           7
37 2025-11-06 15:10:33.501  ask  3347.2    2.2594           3
38 2025-11-06 15:10:33.501  ask  3347.3  370.7208           9
39 2025-11-06 15:10:33.501  ask  3347.4   18.9859          16

Total levels: 40
Bids: 20
Asks: 20
Snapshot time: 2025-11-06 15:10:33.501000


## Mid prices
### hyperliquid_mids

In [ ]:
#| export
def hyperliquid_mids(coin=None,info=None,typecast_to_float=True):
    """
    Retrieves current mid prices from Hyperliquid for specified coins or all available coins.
    
    Args:
        coins (list, optional): List of coin symbols (e.g. ["ETH", "BTC"]). 
            If None, retrieves mids for all available coins. Defaults to None.
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.
    
    Returns:
        dictionary: Dictionary containing mid prices with keys as coin symbols and values as mid prices
        Returns None if the API request fails or returns no data.
    
    Examples:
        # Get mid prices for specific coins
        dic = hyperliquid_mids(info=info)
    
    Notes:
        - This is a snapshot at the moment the function is called
        - Mid price is calculated as (best_bid + best_ask) / 2
        - Requires Hyperliquid Info client to be initialized
    """
    if info is None:
        address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)
    
    try:
        # Get all mid prices from Hyperliquid
        mids_data = info.all_mids()
        
        if not mids_data:
            return None
        if coin:
            if coin not in mids_data.keys():
                return None
            return float(mids_data[coin])
            
        if typecast_to_float: # turn off to save time...
            for i in mids_data.keys():
                mids_data[i] = float(mids_data[i])

        return mids_data
        
    except Exception as e:
        print(f"Error retrieving mid prices: {e}")
        return None


### Example

In [ ]:
#| eval: false
hyperliquid_mids(coin="ETH",info=info)

3344.75

In [ ]:
#| eval: false
hyperliquid_mids(info=info)

{'0G': 1.02345,
 '2Z': 0.16721,
 '@1': 26.3485,
 '@10': 0.00012272,
 '@100': 0.004413,
 '@101': 0.23595,
 '@102': 0.018973,
 '@103': 4.244e-05,
 '@104': 0.070541,
 '@105': 0.39028,
 '@106': 0.017996,
 '@107': 39.488,
 '@108': 0.044976,
 '@109': 0.000413,
 '@11': 0.000721,
 '@110': 0.02005,
 '@111': 0.057739,
 '@112': 0.0006933,
 '@113': 0.0001587,
 '@114': 0.0002464,
 '@115': 0.12255,
 '@116': 2.155e-05,
 '@117': 0.002875,
 '@118': 0.009643,
 '@119': 0.024117,
 '@12': 7.196e-05,
 '@120': 0.030251,
 '@121': 0.010084,
 '@122': 0.019545,
 '@123': 0.10688,
 '@124': 0.00825,
 '@125': 0.013749,
 '@126': 0.21615,
 '@127': 0.091144,
 '@128': 0.001551,
 '@129': 0.016109,
 '@13': 0.00110485,
 '@130': 6.3e-05,
 '@131': 0.07475,
 '@132': 0.003432,
 '@133': 0.000431,
 '@134': 0.001495,
 '@135': 4.1e-05,
 '@136': 0.013192,
 '@137': 2.8e-05,
 '@138': 0.0017217,
 '@139': 0.042716,
 '@14': 9.536e-05,
 '@140': 0.000562,
 '@141': 0.031348,
 '@142': 102368.5,
 '@143': 0.008856,
 '@144': 8.3e-05,
 '@145': 

## Data management

Similar to coinbase.py, this script saves the data in a specified format. These functions handles duplicates and sorts by date. For hourly data, it aligns to the hour.

### HyperliquidDataManager base class

In [ ]:

#| export
def save_hyperliquid_file(df, folder_path, file_name, type="parquet"):
    """
    Save a pandas DataFrame to a file in either CSV or Parquet format.

    Args:
        df (pandas.DataFrame): The DataFrame to save
        folder_path (str): Directory path where the file will be saved
        file_name (str): Name of the file without extension
        type (str, optional): File format - either "csv" or "parquet". Defaults to "parquet"

    The function saves the DataFrame to the specified path, handling the file extension automatically.
    For CSV files, the index is not saved. For Parquet files, default Parquet settings are used.
    Creates the folder if it doesn't exist.
    """
    # Create folder if it doesn't exist
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
    
    if type == "csv":
        df.to_csv(f"{folder_path}/{file_name}.csv", index=False)
    elif type == "parquet":
        df.to_parquet(f"{folder_path}/{file_name}.parquet")
    else:
        raise ValueError(f"Type {type} not supported. Use 'csv' or 'parquet'")

In [ ]:

#| export
class HyperliquidDataManager:
    """
    Base class for managing Hyperliquid data files.
    
    Handles reading, updating, and saving data for different data types (perp, spot, funding).
    Data is stored in organized folders by data type, with files named by token and interval.
    
    Args:
        ticker (str or list, optional): Token symbol(s) to manage. If None, uses all available tokens.
        data_dir (str, optional): Base directory for data storage. Defaults to "../data/hyperliquid"
        interval (str, optional): Time interval for data ("1m", "5m", "15m", "1h", "4h", "1d"). 
            Defaults to "1h". Not used for funding data.
        file_type (str, optional): File format - "parquet" or "csv". Defaults to "parquet"
        update (bool, optional): If True, checks for and downloads new data. Defaults to False
        save (bool, optional): If True, saves updated data back to file. Defaults to False
        refresh_hours (int, optional): Hours of data to refresh when updating. Defaults to 24
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.
        verbose (bool, optional): If True, prints progress messages. Defaults to True
    
    Attributes:
        data_type (str): Type of data managed by this instance ("perp", "spot", "funding")
        data (dict): Dictionary mapping tickers to their DataFrames
    """
    
    def __init__(self, ticker=None, data_dir="../data/hyperliquid", interval="1h",
                 file_type="parquet", update=False, save=False, refresh_hours=24,
                 info=None, verbose=True,data_type=None):
        self.ticker = ticker
        self.data_dir = data_dir
        self.interval = interval
        self.file_type = file_type
        self.update = update
        self.save = save
        self.refresh_hours = refresh_hours
        self.info = info
        self.verbose = verbose
        self.data = {}
        self.data_type = data_type  # To be set by derived classes
        
        # Initialize info client if needed
        if self.info is None:
            if self.update:
                raise ValueError("To update data, Info client must be provided")
        
        # Get list of tickers to process
        self._initialize_tickers()
        
        # Create directory structure if needed
        self._ensure_directories()
    
    def _initialize_tickers(self):
        """Initialize the list of tickers to process."""
        if self.ticker is None:
            # Get all available tokens
            self.tickers = self._get_all_tickers()
        elif isinstance(self.ticker, str):
            self.tickers = [self.ticker]
        elif isinstance(self.ticker, list):
            self.tickers = self.ticker
        else:
            raise ValueError("ticker must be None, str, or list")
    
    def _get_all_tickers(self):
        """Get all available tickers for this data type. To be implemented by derived classes."""
        raise NotImplementedError("Derived classes must implement _get_all_tickers")
    
    def _ensure_directories(self):
        """Create directory structure if it doesn't exist."""
        if self.data_type is None:
            raise ValueError("data_type must be set by derived class")
        
        full_path = os.path.join(self.data_dir, self.data_type)
        if not os.path.exists(full_path):
            os.makedirs(full_path)
            if self.verbose:
                print(f"Created directory: {full_path}")
    
    def _get_file_path(self, ticker):
        """Get the file path for a given ticker."""
        file_name = f"{ticker}_{self.interval}.{self.file_type}" if self.data_type != "funding" else f"{ticker}.{self.file_type}"
        return os.path.join(self.data_dir, self.data_type, file_name)
    
    def _load_existing_data(self, ticker):
        """Load existing data from file if it exists."""
        file_path = self._get_file_path(ticker)
        
        if not os.path.exists(file_path):
            return None
        
        try:
            if self.file_type == "parquet":
                df = pd.read_parquet(file_path)
            elif self.file_type == "csv":
                df = pd.read_csv(file_path)
                df['datetime'] = pd.to_datetime(df['datetime'])
            else:
                raise ValueError(f"Unsupported file type: {self.file_type}")
            
            # Ensure datetime is timezone-aware
            if df['datetime'].dt.tz is None:
                df['datetime'] = pd.to_datetime(df['datetime'], utc=True)
            
            if self.verbose:
                print(f"Loaded {len(df)} rows for {ticker} from {file_path}")
            
            return df
        except Exception as e:
            print(f"Error loading data for {ticker}: {e}")
            return None
    
    def _get_new_data(self, ticker, start_date=None):
        """Retrieve new data from Hyperliquid. To be implemented by derived classes."""
        raise NotImplementedError("Derived classes must implement _get_new_data")
    
    def _update_data(self, ticker, existing_df):
        """Update existing data with new records."""
        import datetime as dt
        
        # Calculate start date for new data
        if existing_df is not None and not existing_df.empty:
            # Get last date and subtract refresh hours
            last_date = pd.to_datetime(existing_df['datetime'].max())
            cutoff_time = last_date - dt.timedelta(hours=self.refresh_hours)
            
            # Remove data within refresh window
            rows_before = len(existing_df)
            existing_df = existing_df[existing_df['datetime'] < cutoff_time]
            rows_removed = rows_before - len(existing_df)
            
            if self.verbose and rows_removed > 0:
                print(f"  Removed {rows_removed} rows from last {self.refresh_hours} hours for refresh")
            
            start_date = cutoff_time.strftime('%Y-%m-%dT%H:%M:%SZ')
        else:
            # No existing data, get default lookback
            start_date = None
        
        # Get new data
        new_df = self._get_new_data(ticker, start_date)
        
        if new_df is None or new_df.empty:
            if self.verbose:
                print(f"  No new data retrieved for {ticker}")
            return existing_df
        
        # Combine with existing data
        if existing_df is not None and not existing_df.empty:
            combined_df = pd.concat([existing_df, new_df], ignore_index=True)
            combined_df = combined_df.drop_duplicates(subset=['datetime'], keep='last')
            combined_df = combined_df.sort_values('datetime').reset_index(drop=True)
            
            if self.verbose:
                print(f"  Updated {ticker}: {len(existing_df)} -> {len(combined_df)} rows")
            
            return combined_df
        else:
            if self.verbose:
                print(f"  Downloaded {len(new_df)} rows for {ticker}")
            return new_df
    
    def _save_data(self, ticker, df):
        """Save data to file."""
        if df is None or df.empty:
            if self.verbose:
                print(f"  No data to save for {ticker}")
            return
        
        file_path = self._get_file_path(ticker)
        
        try:
            if self.file_type == "parquet":
                df.to_parquet(file_path, index=False)
            elif self.file_type == "csv":
                df.to_csv(file_path, index=False)
            else:
                raise ValueError(f"Unsupported file type: {self.file_type}")
            
            if self.verbose:
                print(f"  Saved {len(df)} rows for {ticker} to {file_path}")
        except Exception as e:
            print(f"Error saving data for {ticker}: {e}")
    
    def load_data(self):
        """
        Load data for all tickers.
        
        Returns:
            dict: Dictionary mapping tickers to their DataFrames
        """
        for ticker in self.tickers:
            if self.verbose:
                print(f"Processing {ticker}...")
            
            # Load existing data
            df = self._load_existing_data(ticker)
            
            # Update if requested
            if self.update:
                df = self._update_data(ticker, df)
                
                # Save if requested
                if self.save and df is not None:
                    self._save_data(ticker, df)
            
            # Store in data dictionary
            if df is not None:
                self.data[ticker] = df
        
        return self.data
    
    def get_data(self, ticker=None):
        """
        Get data for specific ticker(s).
        
        Args:
            ticker (str or list, optional): Ticker(s) to retrieve. If None, returns all data.
        
        Returns:
            pandas.DataFrame or dict: DataFrame if single ticker, dict if multiple
        """
        if ticker is None:
            return self.data
        elif isinstance(ticker, str):
            return self.data.get(ticker)
        elif isinstance(ticker, list):
            return {t: self.data.get(t) for t in ticker if t in self.data}
        else:
            raise ValueError("ticker must be None, str, or list")

### HyperliquidPerpManager derived class

In [ ]:

#| export
class HyperliquidPerpManager(HyperliquidDataManager):
    """
    Manager for Hyperliquid perpetual futures data.
    
    Handles reading, updating, and saving perpetual price data (OHLCV).
    Data is stored in the 'perp' subfolder with files named as {ticker}_{interval}.{file_type}
    
    Args:
        ticker (str or list, optional): Token symbol(s) to manage. If None, uses all available perp tokens.
        data_dir (str, optional): Base directory for data storage. Defaults to "../data/hyperliquid"
        interval (str, optional): Time interval for data ("1m", "5m", "15m", "1h", "4h", "1d"). 
            Defaults to "1h".
        file_type (str, optional): File format - "parquet" or "csv". Defaults to "parquet"
        update (bool, optional): If True, checks for and downloads new data. Defaults to False
        save (bool, optional): If True, saves updated data back to file. Defaults to False
        refresh_hours (int, optional): Hours of data to refresh when updating. Defaults to 24
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.
        verbose (bool, optional): If True, prints progress messages. Defaults to True
    
    Examples:
        # Load existing perp data for ETH
        manager = HyperliquidPerpManager(ticker="ETH", interval="1h", info=info)
        eth_data = manager.data["ETH"]
        
        # Update and save data for multiple tokens
        manager = HyperliquidPerpManager(
            ticker=["ETH", "BTC", "SOL"],
            interval="4h",
            update=True,
            save=True,
            refresh_hours=48,
            info=info
        )
        
        # Load all available perp tokens
        manager = HyperliquidPerpManager(update=True, save=True, info=info)
    """
    
    def __init__(self, ticker=None, data_dir="../data/hyperliquid", interval="1h",
                 file_type="parquet", update=False, save=False, refresh_hours=24,
                 info=None, verbose=True):
        # Set data type before calling parent constructor
        self.data_type = "perp"
        
        # Call parent constructor
        super().__init__(ticker=ticker, data_dir=data_dir, interval=interval,
                        file_type=file_type, update=update, save=save,
                        refresh_hours=refresh_hours, info=info, verbose=verbose,data_type="perp")
        
        # Load and optionally update data for all tickers
        self._process_all_tickers()
    
    def _get_all_tokens_from_folder(self):
        """Get all tokens from the perp folder."""
        # Load all files in the perp folder
        file_list = os.listdir(self.data_dir+'/perp')
        #perp_files = [ff for ff in file_list if f"_{self.interval}" in ff)]
        
        # Extract token names from file names
        tickers = [f.split("_")[0] for f in file_list if self.interval in f]
        
        # Return unique tokens
        return list(set(tickers))

    def _get_all_tickers(self):
        """Get all available perpetual tokens from Hyperliquid."""
        try:
            if self.info is not None:
                tickers = hyperliquid_tokens(info=self.info)
                tickers = tickers['name'].tolist()
                if self.verbose:
                    print(f"Found {len(tickers)} perpetual tokens")
                return tickers
            else:
                # read the list of tokens from folder
                tickers = self._get_all_tokens_from_folder()
                if self.verbose:
                    print(f"Found {len(tickers)} perpetual tokens from folder")
                return tickers
        except Exception as e:
            print(f"Error getting perpetual tokens: {e}")
            return []
    
    def _get_new_data(self, ticker, start_date=None):
        """
        Retrieve new perpetual price data from Hyperliquid.
        
        Args:
            ticker (str): Token symbol
            start_date (str, optional): Start date for data retrieval. If None, uses refresh_hours.
        
        Returns:
            pandas.DataFrame: DataFrame with OHLCV data or None if error
        """
        try:
            # Calculate date range
            if start_date is None:
                end_date = None  # Will use current time
                lookback_days = self.refresh_hours / 24
            else:
                end_date = None
                lookback_days = None
            
            # Use retrieve_hyperliquid_data to get perp prices
            df = retrieve_hyperliquid_data(
                ticker=ticker,
                data_type="perp",
                start_date=start_date,
                end_date=end_date,
                lookback=lookback_days if lookback_days else 2,
                interval=self.interval,
                info=self.info
            )
            
            if df is not None and not df.empty:
                if self.verbose:
                    print(f"Retrieved {len(df)} new rows for {ticker}")
            
            return df
            
        except Exception as e:
            print(f"Error retrieving new data for {ticker}: {e}")
            return None
    
    def _update_data(self, ticker, existing_df):
        """
        Update existing perpetual data with new records.
        
        Args:
            ticker (str): Token symbol
            existing_df (pandas.DataFrame): Existing data
        
        Returns:
            pandas.DataFrame: Updated DataFrame with new data merged
        """
        import datetime as dt
        
        # Calculate start date for new data
        if existing_df is not None and not existing_df.empty:
            # Get the most recent datetime from existing data
            max_datetime = existing_df['datetime'].max()
            
            # Subtract refresh_hours to ensure overlap and catch any missing data
            start_datetime = max_datetime - pd.Timedelta(hours=self.refresh_hours)
            start_date = start_datetime.strftime('%Y-%m-%dT%H:%M:%SZ')
            
            if self.verbose:
                print(f"Updating {ticker} from {start_date}")
        else:
            # No existing data, get default lookback period
            start_date = None
            if self.verbose:
                print(f"No existing data for {ticker}, fetching initial data")
        
        # Get new data
        new_df = self._get_new_data(ticker, start_date)
        
        if new_df is None or new_df.empty:
            if self.verbose:
                print(f"No new data retrieved for {ticker}")
            return existing_df
        
        # Merge with existing data
        if existing_df is not None and not existing_df.empty:
            # Combine dataframes
            existing_df['datetime'] = existing_df['datetime'].dt.tz_localize(None)
            combined_df = pd.concat([existing_df, new_df], ignore_index=True)
            # Remove duplicates based on datetime, keeping the last occurrence
            combined_df = combined_df.drop_duplicates(subset=['datetime'], keep='last')
            # Sort by datetime
            combined_df = combined_df.sort_values('datetime').reset_index(drop=True)
            if self.verbose:
                new_rows = len(combined_df) - len(existing_df)
                print(f"Added {new_rows} new rows for {ticker}")
            
            return combined_df
        else:
            # No existing data, return new data
            return new_df.sort_values('datetime').reset_index(drop=True)
    
    def _process_all_tickers(self):
        """Load and optionally update data for all tickers."""
        for ticker in self.tickers:
            try:
                # Load existing data
                existing_df = self._load_existing_data(ticker)
                
                # Update if requested
                if self.update:
                    df = self._update_data(ticker, existing_df)
                else:
                    df = existing_df
                
                # Store in data dictionary
                if df is not None and not df.empty:
                    self.data[ticker] = df
                    
                    # Save if requested
                    if self.save and df is not None:
                        file_path = self._get_file_path(ticker)
                        save_hyperliquid_file(df, os.path.dirname(file_path), 
                                            os.path.splitext(os.path.basename(file_path))[0],
                                            type=self.file_type)
                        if self.verbose:
                            print(f"Saved {len(df)} rows for {ticker}")
                
            except Exception as e:
                print(f"Error processing {ticker}: {e}")
                continue
    
    def get_data(self, ticker=None):
        """
        Get data for a specific ticker.
        
        Args:
            ticker (str): Token symbol
        
        Returns:
            pandas.DataFrame: Data for the ticker or None if not available
        """
        if ticker is None:
            #a = [self.data.get(i) for i in self.tickers]
            return pd.concat(self.data.values())
        return self.data.get(ticker)
    
    def refresh_ticker(self, ticker, save=None):
        """
        Refresh data for a specific ticker.
        
        Args:
            ticker (str): Token symbol
            save (bool, optional): Override instance save setting. If None, uses instance setting.
        
        Returns:
            pandas.DataFrame: Updated data for the ticker
        """
        if ticker not in self.tickers:
            print(f"Ticker {ticker} not in managed tickers")
            return None
        
        # Load existing data
        existing_df = self._load_existing_data(ticker)
        
        # Update data
        df = self._update_data(ticker, existing_df)
        
        # Store in data dictionary
        if df is not None and not df.empty:
            self.data[ticker] = df
            
            # Save if requested
            should_save = save if save is not None else self.save
            if should_save:
                file_path = self._get_file_path(ticker)
                save_hyperliquid_file(df, os.path.dirname(file_path),
                                    os.path.splitext(os.path.basename(file_path))[0],
                                    type=self.file_type)
                if self.verbose:
                    print(f"Saved {len(df)} rows for {ticker}")
        
        return df

#### Example

In [ ]:
#| eval: false
# Load and update ETH perpetual data with 1h interval
manager = HyperliquidPerpManager(
    ticker="ETH",
    interval="1h",
    refresh_hours = 24,
    update=True,
    save=True,
    info=info
)
eth_data = manager.get_data("ETH")
print(eth_data)

Loaded 5569 rows for ETH from ../data/hyperliquid/perp/ETH_1h.parquet
Updating ETH from 2025-11-05T15:00:00Z
Retrieved 26 new rows for ETH
Added 1 new rows for ETH
Saved 5570 rows for ETH
                datetime    open    high     low   close      volume coin
0    2025-03-19 15:00:00  2030.7  2056.7  2026.0  2048.7  30206.2723  ETH
1    2025-03-19 16:00:00  2048.7  2049.2  2035.3  2047.4  22198.4803  ETH
2    2025-03-19 17:00:00  2047.4  2048.9  2014.3  2026.1  35909.8007  ETH
3    2025-03-19 18:00:00  2026.1  2059.9  1998.0  2045.0  84148.3569  ETH
4    2025-03-19 19:00:00  2045.1  2052.1  2020.1  2029.7  38921.5619  ETH
...                  ...     ...     ...     ...     ...         ...  ...
5565 2025-11-06 12:00:00  3398.6  3404.6  3341.7  3348.5  19893.1087  ETH
5566 2025-11-06 13:00:00  3348.5  3402.3  3348.5  3389.8  19783.4383  ETH
5567 2025-11-06 14:00:00  3389.9  3399.8  3318.8  3327.9  32999.1516  ETH
5568 2025-11-06 15:00:00  3327.5  3364.9  3276.7  3299.2  95552.9674  ET

In [ ]:
#|eval: false
manager = HyperliquidPerpManager(
    update=False,
    save=False,
    verbose=False
)
df = manager.get_data()
print(df)


                      datetime     open     high      low    close  \
0    2025-03-28 15:00:00+00:00  0.19480  0.19516  0.19350  0.19369   
1    2025-03-28 16:00:00+00:00  0.19390  0.19470  0.19161  0.19178   
2    2025-03-28 17:00:00+00:00  0.19191  0.19297  0.19191  0.19261   
3    2025-03-28 18:00:00+00:00  0.19279  0.19336  0.19244  0.19296   
4    2025-03-28 19:00:00+00:00  0.19297  0.19318  0.19102  0.19168   
...                        ...      ...      ...      ...      ...   
5348 2025-11-06 10:00:00+00:00  0.32121  0.32329  0.31749  0.31911   
5349 2025-11-06 11:00:00+00:00  0.31924  0.32123  0.31680  0.31923   
5350 2025-11-06 12:00:00+00:00  0.31912  0.31933  0.31349  0.31437   
5351 2025-11-06 13:00:00+00:00  0.31432  0.32621  0.31431  0.32265   
5352 2025-11-06 14:00:00+00:00  0.32292  0.32318  0.31463  0.31783   

          volume coin  
0        25730.0  YGG  
1        11603.0  YGG  
2         9868.0  YGG  
3         9893.0  YGG  
4         9849.0  YGG  
...          ..

### HyperliquidSpotManager derived class

In [ ]:
#| export
class HyperliquidSpotManager(HyperliquidDataManager):
    """
    Manager for Hyperliquid spot market data.
    
    Handles reading, updating, and saving spot price data (OHLCV).
    Data is stored in the 'spot' subfolder with files named as {ticker}_{base}_{interval}.{file_type}
    
    Args:
        ticker (str or list, optional): Token symbol(s) to manage. If None, uses all available spot tokens.
        base (str, optional): Base currency for spot pairs. Defaults to "USDC".
        data_dir (str, optional): Base directory for data storage. Defaults to "../data/hyperliquid"
        interval (str, optional): Time interval for data ("1m", "5m", "15m", "1h", "4h", "1d"). 
            Defaults to "1h".
        file_type (str, optional): File format - "parquet" or "csv". Defaults to "parquet"
        update (bool, optional): If True, checks for and downloads new data. Defaults to False
        save (bool, optional): If True, saves updated data back to file. Defaults to False
        refresh_hours (int, optional): Hours of data to refresh when updating. Defaults to 24
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.
        verbose (bool, optional): If True, prints progress messages. Defaults to True
    
    Examples:
        # Load existing spot data for ETH/USDC
        manager = HyperliquidSpotManager(ticker="ETH", base="USDC", interval="1h", info=info)
        eth_data = manager.data["ETH"]
        
        # Update and save data for multiple tokens
        manager = HyperliquidSpotManager(
            ticker=["ETH", "BTC", "SOL"],
            base="USDC",
            interval="4h",
            update=True,
            save=True,
            refresh_hours=48,
            info=info
        )
        
        # Load all available spot tokens
        manager = HyperliquidSpotManager(base="USDC", update=True, save=True, info=info)
    """
    
    def __init__(self, ticker=None, base="USDC", data_dir="../data/hyperliquid", interval="1h",
                 file_type="parquet", update=False, save=False, refresh_hours=24,
                 info=None, verbose=True):
        # Set data type and base before calling parent constructor
        self.data_type = "spot"
        self.base = base
        
        # Call parent constructor
        super().__init__(ticker=ticker, data_dir=data_dir, interval=interval,
                        file_type=file_type, update=update, save=save,
                        refresh_hours=refresh_hours, info=info, verbose=verbose, data_type="spot")
        
        # Load and optionally update data for all tickers
        self._process_all_tickers()
    
    def _get_file_path(self, ticker):
        """
        Get the file path for a specific ticker, including base currency.
        
        Args:
            ticker (str): Token symbol
        
        Returns:
            str: Full file path
        """
        file_name = f"{ticker}_{self.base}_{self.interval}.{self.file_type}"
        return os.path.join(self.data_dir, self.data_type, file_name)
    
    def _get_all_tickers(self):
        """Get all available spot tokens from Hyperliquid for the specified base."""
        try:
            spot_dict = {'BTC': '@142',
                        'ETH': '@151',
                        'SOL': '@156',
                        'HYPE': '@107',
                        'TRUMP': '@9',
                        'BERA': '@117',
                        'PUMP': '@20'}
            
            tickers = list(spot_dict.keys())
            if self.verbose:
                print(f"Found {len(tickers)} spot tokens for {self.base}")
            return tickers
        except Exception as e:
            print(f"Error getting spot tokens: {e}")
            return []
    
    def _get_new_data(self, ticker, start_date=None):
        """
        Retrieve new spot price data from Hyperliquid.
        
        Args:
            ticker (str): Token symbol
            start_date (str, optional): Start date for data retrieval. If None, uses refresh_hours.
        
        Returns:
            pandas.DataFrame: DataFrame with OHLCV data or None if error
        """
        try:
            # Calculate date range
            if start_date is None:
                end_date = None  # Will use current time
                lookback_days = self.refresh_hours / 24
            else:
                end_date = None
                lookback_days = None
            
            # Use retrieve_hyperliquid_data to get spot prices
            df = retrieve_hyperliquid_data(
                ticker=ticker,
                data_type="spot",
                start_date=start_date,
                end_date=end_date,
                lookback=lookback_days if lookback_days else 2,
                interval=self.interval,
                base=self.base,
                info=self.info
            )
            
            if df is not None and not df.empty:
                if self.verbose:
                    print(f"Retrieved {len(df)} new rows for {ticker}/{self.base}")
            
            return df
            
        except Exception as e:
            print(f"Error retrieving new data for {ticker}/{self.base}: {e}")
            return None
    
    def _update_data(self, ticker, existing_df):
        """
        Update existing spot data with new records.
        
        Args:
            ticker (str): Token symbol
            existing_df (pandas.DataFrame): Existing data
        
        Returns:
            pandas.DataFrame: Updated DataFrame with new data merged
        """
        import datetime as dt
        
        # Calculate start date for new data
        if existing_df is not None and not existing_df.empty:
            # Get the most recent datetime from existing data
            max_datetime = existing_df['datetime'].max()
            
            # Subtract refresh_hours to ensure overlap and catch any missing data
            start_datetime = max_datetime - pd.Timedelta(hours=self.refresh_hours)
            start_date = start_datetime.strftime('%Y-%m-%dT%H:%M:%SZ')
            
            if self.verbose:
                print(f"Updating {ticker}/{self.base} from {start_date}")
        else:
            # No existing data, get default lookback period
            start_date = None
            if self.verbose:
                print(f"No existing data for {ticker}/{self.base}, fetching initial data")
        
        # Get new data
        new_df = self._get_new_data(ticker, start_date)
        
        if new_df is None or new_df.empty:
            if self.verbose:
                print(f"No new data retrieved for {ticker}/{self.base}")
            return existing_df
        
        # Merge with existing data
        if existing_df is not None and not existing_df.empty:
            # Combine dataframes
            existing_df['datetime'] = existing_df['datetime'].dt.tz_localize(None)
            combined_df = pd.concat([existing_df, new_df], ignore_index=True)
            # Remove duplicates based on datetime, keeping the last occurrence
            combined_df = combined_df.drop_duplicates(subset=['datetime'], keep='last')
            # Sort by datetime
            combined_df = combined_df.sort_values('datetime').reset_index(drop=True)
            if self.verbose:
                new_rows = len(combined_df) - len(existing_df)
                print(f"Added {new_rows} new rows for {ticker}/{self.base}")
            
            return combined_df
        else:
            # No existing data, return new data
            return new_df.sort_values('datetime').reset_index(drop=True)
    
    def _process_all_tickers(self):
        """Load and optionally update data for all tickers."""
        for ticker in self.tickers:
            try:
                # Load existing data
                existing_df = self._load_existing_data(ticker)
                
                # Update if requested
                if self.update:
                    df = self._update_data(ticker, existing_df)
                else:
                    df = existing_df
                
                # Store in data dictionary
                if df is not None and not df.empty:
                    self.data[ticker] = df
                    
                    # Save if requested
                    if self.save and df is not None:
                        file_path = self._get_file_path(ticker)
                        save_hyperliquid_file(df, os.path.dirname(file_path), 
                                            os.path.splitext(os.path.basename(file_path))[0],
                                            type=self.file_type)
                        if self.verbose:
                            print(f"Saved {len(df)} rows for {ticker}/{self.base}")
                
            except Exception as e:
                print(f"Error processing {ticker}/{self.base}: {e}")
                continue
    
    def get_data(self, ticker=None):
        """
        Get data for a specific ticker.
        
        Args:
            ticker (str): Token symbol
        
        Returns:
            pandas.DataFrame: Data for the ticker or None if not available
        """
        if ticker is None:
            return pd.concat(self.data.values())
        return self.data.get(ticker)
    
    def refresh_ticker(self, ticker, save=None):
        """
        Refresh data for a specific ticker.
        
        Args:
            ticker (str): Token symbol
            save (bool, optional): Override instance save setting. If None, uses instance setting.
        
        Returns:
            pandas.DataFrame: Updated data for the ticker
        """
        if ticker not in self.tickers:
            print(f"Ticker {ticker} not in managed tickers")
            return None
        
        # Load existing data
        existing_df = self._load_existing_data(ticker)
        
        # Update data
        df = self._update_data(ticker, existing_df)
        
        # Store in data dictionary
        if df is not None and not df.empty:
            self.data[ticker] = df
            
            # Save if requested
            should_save = save if save is not None else self.save
            if should_save:
                file_path = self._get_file_path(ticker)
                save_hyperliquid_file(df, os.path.dirname(file_path),
                                    os.path.splitext(os.path.basename(file_path))[0],
                                    type=self.file_type)
                if self.verbose:
                    print(f"Saved {len(df)} rows for {ticker}/{self.base}")
        
        return df

#### Example

In [ ]:
#| eval: false
# Example 1: Load existing spot data for a single token
print("Example 1: Load existing ETH/USDC spot data")
manager1 = HyperliquidSpotManager(
    ticker="ETH",
    base="USDC",
    interval="1h",
    update=True,
    save=True,
    info=info,
    verbose=True
)
if "ETH" in manager1.data:
    print(f"Loaded {len(manager1.data['ETH'])} records for ETH")
    print(manager1.data["ETH"])

Example 1: Load existing ETH/USDC spot data
Loaded 5167 rows for ETH from ../data/hyperliquid/spot/ETH_USDC_1h.parquet
Updating ETH/USDC from 2025-11-05T14:00:00Z
Retrieved 26 new rows for ETH/USDC
Added 1 new rows for ETH/USDC
Saved 5168 rows for ETH/USDC
Loaded 5168 records for ETH
                datetime    open    high     low   close    volume coin
0    2025-04-05 08:00:00  1812.3  1813.9  1809.7  1812.7    1.7000  ETH
1    2025-04-05 09:00:00  1814.0  1821.0  1814.0  1819.9   19.6808  ETH
2    2025-04-05 10:00:00  1820.2  1820.2  1816.2  1817.4   12.5505  ETH
3    2025-04-05 11:00:00  1818.4  1818.5  1806.6  1806.8   63.9187  ETH
4    2025-04-05 12:00:00  1806.8  1806.9  1793.0  1796.5   45.5064  ETH
...                  ...     ...     ...     ...     ...       ...  ...
5163 2025-11-06 11:00:00  3393.1  3406.3  3381.0  3398.8  127.4400  ETH
5164 2025-11-06 12:00:00  3398.7  3402.6  3342.4  3349.4  141.6060  ETH
5165 2025-11-06 13:00:00  3350.9  3401.8  3349.8  3390.2  119.1310 

In [ ]:
#| eval: false
# Example 2: Update and save data for multiple tokens
print("\nExample 2: Update and save multiple tokens")
manager2 = HyperliquidSpotManager(
    ticker=["ETH", "BTC", "SOL"],
    base="USDC",
    interval="1h",
    update=True,
    save=True,
    refresh_hours=24,
    info=info,
    verbose=True
)

# Access the data
for ticker in ["ETH", "BTC", "SOL"]:
    if ticker in manager2.data:
        print(f"\n{ticker} data shape: {manager2.data[ticker].shape}")
        print(f"Latest {ticker} price: ${manager2.data[ticker]['close'].iloc[-1]:.2f}")


Example 2: Update and save multiple tokens
Loaded 5168 rows for ETH from ../data/hyperliquid/spot/ETH_USDC_1h.parquet
Updating ETH/USDC from 2025-11-05T15:00:00Z
Retrieved 25 new rows for ETH/USDC
Added 0 new rows for ETH/USDC
Saved 5168 rows for ETH/USDC
Loaded 5167 rows for BTC from ../data/hyperliquid/spot/BTC_USDC_1h.parquet
Updating BTC/USDC from 2025-11-05T14:00:00Z
Retrieved 26 new rows for BTC/USDC
Added 1 new rows for BTC/USDC
Saved 5168 rows for BTC/USDC
Loaded 4327 rows for SOL from ../data/hyperliquid/spot/SOL_USDC_1h.parquet
Updating SOL/USDC from 2025-11-05T14:00:00Z
Retrieved 26 new rows for SOL/USDC
Added 1 new rows for SOL/USDC
Saved 4328 rows for SOL/USDC

ETH data shape: (5168, 7)
Latest ETH price: $3345.60

BTC data shape: (5168, 7)
Latest BTC price: $102360.00

SOL data shape: (4328, 7)
Latest SOL price: $158.37


In [ ]:
#| eval: false
# Example 3: Load all available spot tokens with USDC base
print("\nExample 3: Load all available spot tokens")
manager3 = HyperliquidSpotManager(
    base="USDC",
    interval="4h",
    update=True,
    save=True,
    info=info,
    verbose=True
)

print(f"\nLoaded {len(manager3.data)} spot tokens")
print(f"Available tokens: {list(manager3.data.keys())}")


Example 3: Load all available spot tokens
Found 7 spot tokens for USDC
Loaded 49 rows for BTC from ../data/hyperliquid/spot/BTC_USDC_4h.parquet
Updating BTC/USDC from 2025-11-05T12:00:00Z
Retrieved 7 new rows for BTC/USDC
Added 0 new rows for BTC/USDC
Saved 49 rows for BTC/USDC
Loaded 49 rows for ETH from ../data/hyperliquid/spot/ETH_USDC_4h.parquet
Updating ETH/USDC from 2025-11-05T12:00:00Z
Retrieved 7 new rows for ETH/USDC
Added 0 new rows for ETH/USDC
Saved 49 rows for ETH/USDC
Loaded 49 rows for SOL from ../data/hyperliquid/spot/SOL_USDC_4h.parquet
Updating SOL/USDC from 2025-11-05T12:00:00Z
Retrieved 7 new rows for SOL/USDC
Added 0 new rows for SOL/USDC
Saved 49 rows for SOL/USDC
Loaded 49 rows for HYPE from ../data/hyperliquid/spot/HYPE_USDC_4h.parquet
Updating HYPE/USDC from 2025-11-05T12:00:00Z
Retrieved 7 new rows for HYPE/USDC
Added 0 new rows for HYPE/USDC
Saved 49 rows for HYPE/USDC
Loaded 43 rows for TRUMP from ../data/hyperliquid/spot/TRUMP_USDC_4h.parquet
Updating TRUM

In [ ]:
#| eval: false
# Example 5: Analyze spot data
print("\nExample 5: Analyze spot data")
manager5 = HyperliquidSpotManager(
    ticker=["ETH", "BTC"],
    base="USDC",
    interval="1h",
    update=True,
    info=info,
    verbose=True
)

# Calculate some basic statistics
for ticker in ["ETH", "BTC"]:
    if ticker in manager5.data:
        df = manager5.data[ticker]
        
        # Calculate 24h change
        if len(df) >= 24:
            price_24h_ago = df['close'].iloc[-24]
            current_price = df['close'].iloc[-1]
            change_24h = ((current_price - price_24h_ago) / price_24h_ago) * 100
            
            print(f"\n{ticker}/USDC:")
            print(f"  Current Price: ${current_price:.2f}")
            print(f"  24h Change: {change_24h:+.2f}%")
            print(f"  24h High: ${df['high'].iloc[-24:].max():.2f}")
            print(f"  24h Low: ${df['low'].iloc[-24:].min():.2f}")
            print(f"  24h Volume: {df['volume'].iloc[-24:].sum():.2f}")


Example 5: Analyze spot data
Loaded 5168 rows for ETH from ../data/hyperliquid/spot/ETH_USDC_1h.parquet
Updating ETH/USDC from 2025-11-05T15:00:00Z
Retrieved 25 new rows for ETH/USDC
Added 0 new rows for ETH/USDC
Loaded 5168 rows for BTC from ../data/hyperliquid/spot/BTC_USDC_1h.parquet
Updating BTC/USDC from 2025-11-05T15:00:00Z
Retrieved 25 new rows for BTC/USDC
Added 0 new rows for BTC/USDC

ETH/USDC:
  Current Price: $3344.60
  24h Change: -2.74%
  24h High: $3480.00
  24h Low: $3320.00
  24h Volume: 6410.37

BTC/USDC:
  Current Price: $102360.00
  24h Change: -1.54%
  24h High: $104540.00
  24h Low: $101900.00
  24h Volume: 1082.17


In [ ]:
#|eval: false
mananger6 = HyperliquidSpotManager(
    base="USDC",
    interval="1h",
    update=False,
    save=False,
    verbose=True
)
mananger6.get_data()

Found 7 spot tokens for USDC
Loaded 5168 rows for BTC from ../data/hyperliquid/spot/BTC_USDC_1h.parquet
Loaded 5168 rows for ETH from ../data/hyperliquid/spot/ETH_USDC_1h.parquet
Loaded 4328 rows for SOL from ../data/hyperliquid/spot/SOL_USDC_1h.parquet
Loaded 164 rows for HYPE from ../data/hyperliquid/spot/HYPE_USDC_1h.parquet
Loaded 154 rows for PUMP from ../data/hyperliquid/spot/PUMP_USDC_1h.parquet


,datetime,open,high,low,close,volume,coin
0,2025-04-05 08:00:00+00:00,83511.000000,83623.000000,83361.000000,83482.000000,2.416920e+00,BTC
1,2025-04-05 09:00:00+00:00,83486.000000,83854.000000,83486.000000,83848.000000,6.817760e+00,BTC
2,2025-04-05 10:00:00+00:00,83855.000000,83901.000000,83600.000000,83642.000000,6.209230e+00,BTC
3,2025-04-05 11:00:00+00:00,83642.000000,83688.000000,83402.000000,83430.000000,9.042830e+00,BTC
4,2025-04-05 12:00:00+00:00,83441.000000,83517.000000,83112.000000,83243.000000,6.552230e+00,BTC
...,...,...,...,...,...,...,...
149,2025-11-05 23:00:00+00:00,0.000139,0.000139,0.000139,0.000139,0.000000e+00,PUMP
150,2025-11-06 00:00:00+00:00,0.000139,0.000139,0.000139,0.000139,0.000000e+00,PUMP
151,2025-11-06 01:00:00+00:00,0.000139,0.000145,0.000133,0.000133,6.914432e+06,PUMP
152,2025-11-06 02:00:00+00:00,0.000139,0.000145,0.000139,0.000139,1.384991e+07,PUMP


### HyperliquidFundingManager derived class

In [ ]:
#| export
class HyperliquidFundingManager(HyperliquidDataManager):
    """
    Manager for Hyperliquid funding rate data.
    
    Handles reading, updating, and saving funding rate data.
    Data is stored in the 'funding' subfolder with files named as {ticker}.{file_type}
    
    Args:
        ticker (str or list, optional): Token symbol(s) to manage. If None, uses all available tokens.
        data_dir (str, optional): Base directory for data storage. Defaults to "../data/hyperliquid"
        file_type (str, optional): File format - "parquet" or "csv". Defaults to "parquet"
        update (bool, optional): If True, checks for and downloads new data. Defaults to False
        save (bool, optional): If True, saves updated data back to file. Defaults to False
        refresh_hours (int, optional): Hours of data to refresh when updating. Defaults to 24
        round_to_hour (bool, optional): If True, rounds datetime to nearest hour. Defaults to True
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.
        verbose (bool, optional): If True, prints progress messages. Defaults to True
    
    Examples:
        # Load existing funding data for ETH
        manager = HyperliquidFundingManager(ticker="ETH", info=info)
        eth_data = manager.data["ETH"]
        
        # Update and save data for multiple tokens
        manager = HyperliquidFundingManager(
            ticker=["ETH", "BTC", "SOL"],
            update=True,
            save=True,
            refresh_hours=48,
            info=info
        )
        
        # Load all available tokens
        manager = HyperliquidFundingManager(update=True, save=True, info=info)
    """
    
    def __init__(self, ticker=None, data_dir="../data/hyperliquid", 
                 file_type="parquet", update=False, save=False, refresh_hours=24,
                 round_to_hour=True, info=None, verbose=True):
        # Set data type and round_to_hour before calling parent constructor
        self.data_type = "funding"
        self.round_to_hour = round_to_hour
        
        # Call parent constructor (interval not used for funding data, but required by parent)
        super().__init__(ticker=ticker, data_dir=data_dir, interval="1h",
                        file_type=file_type, update=update, save=save,
                        refresh_hours=refresh_hours, info=info, verbose=verbose,
                        data_type="funding")
        
        # Load and optionally update data for all tickers
        self._process_all_tickers()
    
    def _get_all_tokens_from_folder(self):
        """Get all tokens from the perp folder."""
        # Load all files in the perp folder
        file_list = os.listdir(self.data_dir+'/funding')
        
        # Extract token names from file names
        tickers = [f.split(".")[0] for f in file_list]
        
        # Return unique tokens
        return list(set(tickers))

    def _get_all_tickers(self):
        """Get all available tokens from Hyperliquid."""
        try:
            if self.info is None:
                # read the list of tokens from folder
                tickers = self._get_all_tokens_from_folder()
                if self.verbose:
                    print(f"Found {len(tickers)} perpetual tokens from folder")
                return tickers
            tickers = hyperliquid_tokens(info=self.info)
            tickers = tickers['name'].tolist()
            if self.verbose:
                print(f"Found {len(tickers)} tokens")
            return tickers
        except Exception as e:
            print(f"Error getting tokens: {e}")
            return []
    
    def _get_file_path(self, ticker):
        """Get the file path for a given ticker (funding data doesn't use interval in filename)."""
        file_name = f"{ticker}.{self.file_type}"
        return os.path.join(self.data_dir, self.data_type, file_name)
    
    def _get_new_data(self, ticker, start_date=None):
        """
        Retrieve new funding rate data from Hyperliquid.
        
        Args:
            ticker (str): Token symbol
            start_date (str, optional): Start date for data retrieval. If None, uses refresh_hours.
        
        Returns:
            pandas.DataFrame: DataFrame with funding rate data or None if error
        """
        try:
            # Calculate date range
            if start_date is None:
                end_date = None  # Will use current time
                lookback_days = self.refresh_hours / 24
            else:
                end_date = None
                lookback_days = None
            
            # Use retrieve_hyperliquid_data to get funding rates
            df = retrieve_hyperliquid_data(
                ticker=ticker,
                data_type="funding",
                start_date=start_date,
                end_date=end_date,
                lookback=lookback_days if lookback_days else 2,
                round_to_hour=self.round_to_hour,
                info=self.info
            )
            
            if df is not None and not df.empty:
                if self.verbose:
                    print(f"Retrieved {len(df)} new rows for {ticker}")
            
            return df
            
        except Exception as e:
            print(f"Error retrieving new data for {ticker}: {e}")
            return None
    
    def _update_data(self, ticker, existing_df):
        """
        Update existing funding rate data with new records.
        
        Args:
            ticker (str): Token symbol
            existing_df (pandas.DataFrame): Existing data
        
        Returns:
            pandas.DataFrame: Updated DataFrame with new data merged
        """
        import datetime as dt
        
        # Calculate start date for new data
        if existing_df is not None and not existing_df.empty:
            # Get the most recent datetime from existing data
            max_datetime = existing_df['datetime'].max()
            
            # Subtract refresh_hours to ensure overlap and catch any missing data
            start_datetime = max_datetime - pd.Timedelta(hours=self.refresh_hours)
            start_date = start_datetime.strftime('%Y-%m-%dT%H:%M:%SZ')
            
            if self.verbose:
                print(f"Updating {ticker} from {start_date}")
        else:
            # No existing data, get default lookback period
            start_date = None
            if self.verbose:
                print(f"No existing data for {ticker}, fetching initial data")
        
        # Get new data
        new_df = self._get_new_data(ticker, start_date)
        
        if new_df is None or new_df.empty:
            if self.verbose:
                print(f"No new data retrieved for {ticker}")
            return existing_df
        
        # Merge with existing data
        if existing_df is not None and not existing_df.empty:
            # Combine dataframes
            existing_df['datetime'] = existing_df['datetime'].dt.tz_localize(None)
            combined_df = pd.concat([existing_df, new_df], ignore_index=True)
            # Remove duplicates based on datetime, keeping the last occurrence
            combined_df = combined_df.drop_duplicates(subset=['datetime'], keep='last')
            # Sort by datetime
            combined_df = combined_df.sort_values('datetime').reset_index(drop=True)
            if self.verbose:
                new_rows = len(combined_df) - len(existing_df)
                print(f"Added {new_rows} new rows for {ticker}")
            
            return combined_df
        else:
            # No existing data, return new data
            return new_df.sort_values('datetime').reset_index(drop=True)
    
    def _process_all_tickers(self):
        """Load and optionally update data for all tickers."""
        for ticker in self.tickers:
            try:
                # Load existing data
                existing_df = self._load_existing_data(ticker)
                
                # Update if requested
                if self.update:
                    df = self._update_data(ticker, existing_df)
                else:
                    df = existing_df
                
                # Store in data dictionary
                if df is not None and not df.empty:
                    self.data[ticker] = df
                    
                    # Save if requested
                    if self.save and df is not None:
                        file_path = self._get_file_path(ticker)
                        save_hyperliquid_file(df, os.path.dirname(file_path), 
                                            os.path.splitext(os.path.basename(file_path))[0],
                                            type=self.file_type)
                        if self.verbose:
                            print(f"Saved {len(df)} rows for {ticker}")
                
            except Exception as e:
                print(f"Error processing {ticker}: {e}")
                continue
    
    def get_data(self, ticker=None):
        """
        Get data for a specific ticker.
        
        Args:
            ticker (str): Token symbol
        
        Returns:
            pandas.DataFrame: Data for the ticker or None if not available
        """
        if ticker is None:
            return pd.concat(self.data.values())
        return self.data.get(ticker)
    
    def refresh_ticker(self, ticker, save=None):
        """
        Refresh data for a specific ticker.
        
        Args:
            ticker (str): Token symbol
            save (bool, optional): Override instance save setting. If None, uses instance setting.
        
        Returns:
            pandas.DataFrame: Updated data for the ticker
        """
        if ticker not in self.tickers:
            print(f"Ticker {ticker} not in managed tickers")
            return None
        
        # Load existing data
        existing_df = self._load_existing_data(ticker)
        
        # Update data
        df = self._update_data(ticker, existing_df)
        
        # Store in data dictionary
        if df is not None and not df.empty:
            self.data[ticker] = df
            
            # Save if requested
            should_save = save if save is not None else self.save
            if should_save:
                file_path = self._get_file_path(ticker)
                save_hyperliquid_file(df, os.path.dirname(file_path),
                                    os.path.splitext(os.path.basename(file_path))[0],
                                    type=self.file_type)
                if self.verbose:
                    print(f"Saved {len(df)} rows for {ticker}")
        
        return df

#### Example

In [ ]:
#|eval:false
manager = HyperliquidFundingManager(ticker="ETH")
eth_data = manager.data["ETH"]
print(eth_data)

Loaded 430 rows for ETH from ../data/hyperliquid/funding/ETH.parquet
                     datetime  funding_rate   premium coin  fund_calc
0   2025-10-03 21:00:00+00:00      0.000013  0.000185  ETH   0.000013
1   2025-10-03 22:00:00+00:00      0.000013  0.000111  ETH   0.000013
2   2025-10-03 23:00:00+00:00      0.000013  0.000141  ETH   0.000013
3   2025-10-04 00:00:00+00:00      0.000013  0.000113  ETH   0.000013
4   2025-10-04 01:00:00+00:00      0.000013  0.000081  ETH   0.000013
..                        ...           ...       ...  ...        ...
425 2025-11-06 11:00:00+00:00      0.000013 -0.000206  ETH   0.000013
426 2025-11-06 12:00:00+00:00      0.000013 -0.000178  ETH   0.000013
427 2025-11-06 13:00:00+00:00      0.000013 -0.000255  ETH   0.000013
428 2025-11-06 14:00:00+00:00      0.000013 -0.000284  ETH   0.000013
429 2025-11-06 15:00:00+00:00      0.000013 -0.000379  ETH   0.000013

[430 rows x 5 columns]


In [ ]:
#|eval:false
manager = HyperliquidFundingManager(update=False, save=False,verbose=False)
manager.get_data()

,datetime,funding_rate,premium,coin,fund_calc
0,2025-10-03 21:00:00+00:00,0.000013,0.000117,YGG,0.000013
1,2025-10-03 22:00:00+00:00,0.000013,0.000146,YGG,0.000013
2,2025-10-03 23:00:00+00:00,0.000013,0.000110,YGG,0.000013
3,2025-10-04 00:00:00+00:00,0.000013,0.000198,YGG,0.000013
4,2025-10-04 01:00:00+00:00,0.000013,0.000148,YGG,0.000013
...,...,...,...,...,...
425,2025-11-06 11:00:00+00:00,0.000008,-0.000436,ENA,0.000008
426,2025-11-06 12:00:00+00:00,0.000006,-0.000450,ENA,0.000006
427,2025-11-06 13:00:00+00:00,0.000008,-0.000433,ENA,0.000008
428,2025-11-06 14:00:00+00:00,0.000003,-0.000478,ENA,0.000003


## More examples

Load package:

In [ ]:
#|eval:false
from token_data.hyperliquid import *

Load all available funding rates already stored in the data directory. It returns a pandas DataFrame with columns 'datetime' in UTC and 'funding_rate' as the funding rate used within the Hyperliquid system to find the USDC paid or received for a given perpetual position.

The other coluns are "premium" and "fund_calc". The "premium" column represents the premium used in the funding rate calculation and the "fund_calc" is currently very experimental and tries to replicated the funding rate calculation in the Hyperliquid system. Use "funding_rate" column in simulations or P&L calculations.

In [ ]:
#|eval:false
manager = HyperliquidFundingManager(update=False, save=False, verbose=False)
manager.get_data()

,datetime,funding_rate,premium,coin,fund_calc
0,2025-10-03 21:00:00+00:00,0.000013,0.000117,YGG,0.000013
1,2025-10-03 22:00:00+00:00,0.000013,0.000146,YGG,0.000013
2,2025-10-03 23:00:00+00:00,0.000013,0.000110,YGG,0.000013
3,2025-10-04 00:00:00+00:00,0.000013,0.000198,YGG,0.000013
4,2025-10-04 01:00:00+00:00,0.000013,0.000148,YGG,0.000013
...,...,...,...,...,...
425,2025-11-06 11:00:00+00:00,0.000008,-0.000436,ENA,0.000008
426,2025-11-06 12:00:00+00:00,0.000006,-0.000450,ENA,0.000006
427,2025-11-06 13:00:00+00:00,0.000008,-0.000433,ENA,0.000008
428,2025-11-06 14:00:00+00:00,0.000003,-0.000478,ENA,0.000003


Load all available spot prices already stored in the data directory. It returns a pandas DataFrame with columns 'datetime' in UTC and 'open', 'high', 'low', 'close', 'volume' and 'coin'. Important to note that the list of spots is limited and was hard coded. In the future this can be changed, therefore, the list is limited to some of the most important tokens.

In [ ]:
#|eval:false
manager = HyperliquidSpotManager(update=False, save=False, verbose=False)
manager.get_data()

,datetime,open,high,low,close,volume,coin
0,2025-04-05 08:00:00+00:00,83511.000000,83623.000000,83361.000000,83482.000000,2.416920e+00,BTC
1,2025-04-05 09:00:00+00:00,83486.000000,83854.000000,83486.000000,83848.000000,6.817760e+00,BTC
2,2025-04-05 10:00:00+00:00,83855.000000,83901.000000,83600.000000,83642.000000,6.209230e+00,BTC
3,2025-04-05 11:00:00+00:00,83642.000000,83688.000000,83402.000000,83430.000000,9.042830e+00,BTC
4,2025-04-05 12:00:00+00:00,83441.000000,83517.000000,83112.000000,83243.000000,6.552230e+00,BTC
...,...,...,...,...,...,...,...
149,2025-11-05 23:00:00+00:00,0.000139,0.000139,0.000139,0.000139,0.000000e+00,PUMP
150,2025-11-06 00:00:00+00:00,0.000139,0.000139,0.000139,0.000139,0.000000e+00,PUMP
151,2025-11-06 01:00:00+00:00,0.000139,0.000145,0.000133,0.000133,6.914432e+06,PUMP
152,2025-11-06 02:00:00+00:00,0.000139,0.000145,0.000139,0.000139,1.384991e+07,PUMP


Next example loads perpetual prices already stored in the data directory on an hourly frequency (default but can be changed). It returns a pandas DataFrame with columns 'datetime' in UTC and 'open', 'high', 'low', 'close', 'volume' and 'coin'.

In [ ]:
#|eval:false
manager = HyperliquidPerpManager(update=False, save=False,verbose=False)
manager.get_data()

,datetime,open,high,low,close,volume,coin
0,2025-03-28 15:00:00+00:00,0.19480,0.19516,0.19350,0.19369,25730.0,YGG
1,2025-03-28 16:00:00+00:00,0.19390,0.19470,0.19161,0.19178,11603.0,YGG
2,2025-03-28 17:00:00+00:00,0.19191,0.19297,0.19191,0.19261,9868.0,YGG
3,2025-03-28 18:00:00+00:00,0.19279,0.19336,0.19244,0.19296,9893.0,YGG
4,2025-03-28 19:00:00+00:00,0.19297,0.19318,0.19102,0.19168,9849.0,YGG
...,...,...,...,...,...,...,...
5348,2025-11-06 10:00:00+00:00,0.32121,0.32329,0.31749,0.31911,8988547.0,ENA
5349,2025-11-06 11:00:00+00:00,0.31924,0.32123,0.31680,0.31923,3874363.0,ENA
5350,2025-11-06 12:00:00+00:00,0.31912,0.31933,0.31349,0.31437,6577452.0,ENA
5351,2025-11-06 13:00:00+00:00,0.31432,0.32621,0.31431,0.32265,23799351.0,ENA


Next example loads only "ETH" perpetual prices already stored in the data directory on an hourly frequency (default but can be changed). It returns a pandas DataFrame with columns 'datetime' in UTC.

In [ ]:
#|eval:false
manager = HyperliquidPerpManager(ticker='ETH',update=False, save=False,verbose=True)
manager.get_data()

Loaded 5570 rows for ETH from ../data/hyperliquid/perp/ETH_1h.parquet


,datetime,open,high,low,close,volume,coin
0,2025-03-19 15:00:00+00:00,2030.7,2056.7,2026.0,2048.7,30206.2723,ETH
1,2025-03-19 16:00:00+00:00,2048.7,2049.2,2035.3,2047.4,22198.4803,ETH
2,2025-03-19 17:00:00+00:00,2047.4,2048.9,2014.3,2026.1,35909.8007,ETH
3,2025-03-19 18:00:00+00:00,2026.1,2059.9,1998.0,2045.0,84148.3569,ETH
4,2025-03-19 19:00:00+00:00,2045.1,2052.1,2020.1,2029.7,38921.5619,ETH
...,...,...,...,...,...,...,...
5565,2025-11-06 12:00:00+00:00,3398.6,3404.6,3341.7,3348.5,19893.1087,ETH
5566,2025-11-06 13:00:00+00:00,3348.5,3402.3,3348.5,3389.8,19783.4383,ETH
5567,2025-11-06 14:00:00+00:00,3389.9,3399.8,3318.8,3327.9,32999.1516,ETH
5568,2025-11-06 15:00:00+00:00,3327.5,3364.9,3276.7,3299.2,95552.9674,ETH


If you want to make sure you are featching the most recent data, use the `update=True` parameter. In that case one needs to pass the `info` parameter with your API key to retrieve real-time data which is setup beforehand. The next example demonstrates how to use this:

In [ ]:
#|eval:false
address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)
manager = HyperliquidPerpManager(ticker='ETH',update=True, save=False, verbose=True,info=info)
manager.get_data()

Loaded 5570 rows for ETH from ../data/hyperliquid/perp/ETH_1h.parquet
Updating ETH from 2025-11-05T16:00:00Z
Retrieved 25 new rows for ETH
Added 0 new rows for ETH


,datetime,open,high,low,close,volume,coin
0,2025-03-19 15:00:00,2030.7,2056.7,2026.0,2048.7,30206.2723,ETH
1,2025-03-19 16:00:00,2048.7,2049.2,2035.3,2047.4,22198.4803,ETH
2,2025-03-19 17:00:00,2047.4,2048.9,2014.3,2026.1,35909.8007,ETH
3,2025-03-19 18:00:00,2026.1,2059.9,1998.0,2045.0,84148.3569,ETH
4,2025-03-19 19:00:00,2045.1,2052.1,2020.1,2029.7,38921.5619,ETH
...,...,...,...,...,...,...,...
5565,2025-11-06 12:00:00,3398.6,3404.6,3341.7,3348.5,19893.1087,ETH
5566,2025-11-06 13:00:00,3348.5,3402.3,3348.5,3389.8,19783.4383,ETH
5567,2025-11-06 14:00:00,3389.9,3399.8,3318.8,3327.9,32999.1516,ETH
5568,2025-11-06 15:00:00,3327.5,3364.9,3276.7,3299.2,95552.9674,ETH


The same goes for the other data items. Such as spot prices and funding rates. Next example shows the spot prices for "ETH" on an hourly frequency and updates it:

In [ ]:
#|eval:false
manager = HyperliquidSpotManager(ticker='ETH',update=True, save=False, verbose=True,info=info)
manager.get_data()

Loaded 5168 rows for ETH from ../data/hyperliquid/spot/ETH_USDC_1h.parquet
Updating ETH/USDC from 2025-11-05T15:00:00Z
Retrieved 26 new rows for ETH/USDC
Added 1 new rows for ETH/USDC


,datetime,open,high,low,close,volume,coin
0,2025-04-05 08:00:00,1812.3,1813.9,1809.7,1812.7,1.7000,ETH
1,2025-04-05 09:00:00,1814.0,1821.0,1814.0,1819.9,19.6808,ETH
2,2025-04-05 10:00:00,1820.2,1820.2,1816.2,1817.4,12.5505,ETH
3,2025-04-05 11:00:00,1818.4,1818.5,1806.6,1806.8,63.9187,ETH
4,2025-04-05 12:00:00,1806.8,1806.9,1793.0,1796.5,45.5064,ETH
...,...,...,...,...,...,...,...
5164,2025-11-06 12:00:00,3398.7,3402.6,3342.4,3349.4,141.6060,ETH
5165,2025-11-06 13:00:00,3350.9,3401.8,3349.8,3390.2,119.1310,ETH
5166,2025-11-06 14:00:00,3390.1,3398.5,3320.0,3327.1,396.8991,ETH
5167,2025-11-06 15:00:00,3327.6,3363.2,3278.9,3298.1,523.0398,ETH


If you want to update the funding rate, do the following:

In [ ]:
#|eval:false
manager = HyperliquidFundingManager(ticker='ETH',update=True, save=False, verbose=True,info=info)
manager.get_data()

Loaded 430 rows for ETH from ../data/hyperliquid/funding/ETH.parquet
Updating ETH from 2025-11-05T15:00:00Z
Retrieved 26 new rows for ETH
Added 1 new rows for ETH


,datetime,funding_rate,premium,coin,fund_calc
0,2025-10-03 21:00:00,0.000013,0.000185,ETH,0.000013
1,2025-10-03 22:00:00,0.000013,0.000111,ETH,0.000013
2,2025-10-03 23:00:00,0.000013,0.000141,ETH,0.000013
3,2025-10-04 00:00:00,0.000013,0.000113,ETH,0.000013
4,2025-10-04 01:00:00,0.000013,0.000081,ETH,0.000013
...,...,...,...,...,...
426,2025-11-06 12:00:00,0.000013,-0.000178,ETH,0.000013
427,2025-11-06 13:00:00,0.000013,-0.000255,ETH,0.000013
428,2025-11-06 14:00:00,0.000013,-0.000284,ETH,0.000013
429,2025-11-06 15:00:00,0.000013,-0.000379,ETH,0.000013
